# DSA 4060 – Week 3 Practical Lab
## Building a Simple Content-Based Recommender

**Student:** Halima Mahdi  
**Course:** DSA 4060 – Recommender Systems  
**Topic:** Content-Based Filtering Using Structured Features

### Lab Objective

This notebook implements a simple content-based recommender using structured movie metadata. Movies are represented by binary genre features, while a user's ratings are used to construct a weighted preference profile. The system then scores unseen movies, removes movies already watched, ranks the remaining candidates, and returns Top-N recommendations.

### Workflow

**Movie Metadata → Item Profiles → User Ratings → User Profile → Recommendation Scores → Remove Seen Items → Rank Items → Top-N Recommendations**

> **Note:** This Week 3 implementation intentionally uses structured features only. TF-IDF and cosine similarity are not used because those topics are reserved for Week 4.

## 1. Import Libraries

We use **pandas** for creating, loading, inspecting, transforming, and analysing the movie and rating datasets.

The notebook does not require a machine-learning library because the recommender is built directly from structured feature values and user ratings.

In [ ]:
import pandas as pd
from IPython.display import display

## 2. Create / Load the Movie Dataset

The lab requires a movie dataset containing a unique movie identifier, a movie title, and structured binary genre features.

Instead of copying the sample dataset supplied in the practical instructions, this project uses an original 15-movie dataset created specifically for this notebook.

Each genre feature is encoded as:

- **1** = the movie contains that genre
- **0** = the movie does not contain that genre

The five structured features are **Action, Comedy, Drama, Romance, and SciFi**.

In [ ]:
movies = pd.read_csv("movies.csv")
movies

## 3. Explore the Dataset

Before building a recommender, it is important to understand the structure of the data.

The following cells inspect the first rows, number of rows and columns, and data types.

In [ ]:
display(movies.head())
print("Dataset shape:", movies.shape)
movies.info()

### Dataset Exploration Answers

**1. How many movies are in the dataset?**  
There are **15 movies**.

**2. How many genre features exist?**  
There are **5 genre features**: Action, Comedy, Drama, Romance, and SciFi.

**3. What does one row represent?**  
One row represents **one movie and its structured genre profile**. The binary genre columns describe which types of content are present in that movie.

## 4. Identify the Structured Features

The `movie_id` and `title` columns identify each item. The five genre columns are the structured features used by the recommender.

These features form the **item-feature matrix**.

In [ ]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]

movie_features = movies[features]

movie_features

## 5. Inspect One Item Profile

An item profile describes a movie using its feature values.

For example, the profile for **Midnight Chase** contains `Action = 1` while its other genre values indicate whether Comedy, Drama, Romance, or SciFi are present.

A value of **1** means that the feature is present in the movie, while **0** means that the feature is absent.

In [ ]:
movies.loc[movies["title"] == "Midnight Chase"]

### Interpretation

The binary values in the selected movie's profile show which genres are associated with it. This allows the recommender to compare a user's preferred genres with the genres contained in candidate movies.

## 6. Create User Ratings

We now create a rating history for **User A**.

The ratings represent how strongly the user liked four movies:

| Movie | Rating |
|---|---:|
| Midnight Chase | 5 |
| Nairobi Love Story | 2 |
| The Last Guardian | 4 |
| Orbit Beyond Earth | 5 |

A higher rating means the movie contributes more strongly to the user's preference profile.

In [ ]:
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})

ratings

## 7. Join Ratings With Movie Features

The ratings dataset only contains movie IDs and ratings. We merge it with the movie dataset so that every rating can be examined together with the corresponding genre features.

In [ ]:
user_movies = ratings.merge(
    movies,
    on="movie_id"
)

user_movies

## 8. Examine the User's History

The following table focuses on the information that matters for preference modelling.

In [ ]:
user_movies[
    ["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]
]

### Observation

User A appears to have a strong interest in **Action and SciFi-related content**. The highest ratings are given to *Midnight Chase* and *Orbit Beyond Earth*, both of which contain Action/SciFi-related features. The lower rating for *Nairobi Love Story* also suggests that Romance is not as strong a preference for this user.

## 9. Select the Rated Features

We separate the genre features from the ratings so that each movie's feature vector can be weighted according to the user's rating.

In [ ]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]

display(rated_features)
display(user_ratings)

## 10. Weight Features Using Ratings

A highly rated movie should have a stronger influence on the user's preference profile than a poorly rated movie.

We therefore multiply each feature vector by the corresponding rating.

For example, if a movie has `Action = 1` and the user gives it a rating of 5, its contribution to the Action preference is 5.

In [ ]:
weighted_features = rated_features.mul(
    user_ratings,
    axis=0
)

weighted_features

## 11. Build the User Preference Profile

The weighted feature values are summed across all movies rated by the user.

The resulting vector represents the user's overall preference strength for each genre.

In [ ]:
user_profile = weighted_features.sum()

user_profile

## 12. Interpret the User Profile

Sorting the profile from strongest to weakest makes the user's preferences easier to understand.

In [ ]:
user_profile.sort_values(ascending=False)

### Interpretation

The strongest preferences for User A are **Action and SciFi**.

This is because the user's highest ratings are attached to movies containing those features. Drama also receives a relatively strong contribution, while Romance and Comedy have weaker or no contributions in the rated history.

## 13. Normalize the User Profile

The raw profile values are converted into proportions by dividing every feature score by the total profile score.

Normalization makes the profile easier to interpret because the feature weights sum to 1.

In [ ]:
user_profile_normalized = user_profile / user_profile.sum()

user_profile_normalized

## 14. Score Every Movie

A movie's recommendation score is calculated by multiplying its binary feature vector by the normalized user profile and summing the results.

A higher score means the movie contains more of the features that are important to the user.

In [ ]:
movies["recommendation_score"] = (
    movie_features
    .mul(user_profile_normalized)
    .sum(axis=1)
)

movies[["title", "recommendation_score"]]

## 15. Rank the Movies

Movies are sorted from the highest recommendation score to the lowest score.

In [ ]:
ranked_movies = movies.sort_values(
    "recommendation_score",
    ascending=False
)

ranked_movies[["title", "recommendation_score"]]

### Interpretation

The ranking is based on how closely the structured genre content of each movie matches the user's learned preference profile. A movie with more strongly preferred features receives a higher score.

## 16. Remove Movies Already Watched

A recommender should not recommend movies that the user has already rated.

We therefore remove all movie IDs that appear in the user's rating history.

In [ ]:
watched_movies = ratings["movie_id"]

recommendations = ranked_movies[
    ~ranked_movies["movie_id"].isin(watched_movies)
]

recommendations[["title", "recommendation_score"]]

## 17. Generate Top 3 Recommendations

The first three unseen movies in the ranked list are returned as the Top 3 recommendations.

In [ ]:
top_3 = recommendations[
    ["title", "recommendation_score"]
].head(3)

top_3

## 18. Explain the Top 3 Recommendations

The recommendations should be interpreted using the user's strongest preferences rather than treating the score as a black box.

For User A, **Action and SciFi** are the strongest preferences. Therefore, movies containing those features are expected to receive relatively high scores.

The exact ranking is generated by the code above, so the recommendation explanations should be read together with the displayed scores.

## 19. Build a Reusable Recommendation Function

The previous steps can be combined into one reusable function.

The function:
1. Joins the user's ratings with movie metadata.
2. Extracts the structured features.
3. Weights features by ratings.
4. Builds and normalizes the user profile.
5. Calculates recommendation scores.
6. Removes watched movies.
7. Sorts the remaining movies.
8. Returns the requested number of recommendations.

In [ ]:
def recommend_movies(movies, ratings, features, n=3):
    user_movies = ratings.merge(movies, on="movie_id")

    rated_features = user_movies[features]

    weighted_features = rated_features.mul(
        user_movies["rating"],
        axis=0
    )

    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()

    scores = (
        movies[features]
        .mul(user_profile)
        .sum(axis=1)
    )

    results = movies.copy()
    results["recommendation_score"] = scores

    watched = ratings["movie_id"]

    results = results[
        ~results["movie_id"].isin(watched)
    ]

    return results.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)

## 20. Test the Reusable Function With User A

We now use the function to generate recommendations for User A.

In [ ]:
recommend_movies(
    movies,
    ratings,
    features,
    n=3
)

## 21. Test a Second User

A content-based recommender should be able to produce different results when users have different rating histories.

User B has rated four different movies with a different pattern of ratings.

In [ ]:
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})

user_b_ratings

In [ ]:
recommend_movies(
    movies,
    user_b_ratings,
    features,
    n=3
)

### User A vs User B

The users do not necessarily receive the same recommendations because their ratings create different preference profiles.

User A's history gives stronger evidence for Action and SciFi, while User B's history places greater weight on the features represented by the movies they rated highly, particularly Romance, Comedy, and Drama.

Therefore, even though the recommender uses the same movie catalogue, the user profile changes the recommendation scores.

## 22. Thinking Exercise: User C

Suppose User C has watched only **Midnight Chase** and rated it 5 stars.

### What will the model learn?

The model will learn a very limited profile based almost entirely on the features of *Midnight Chase*.

### Which genres will dominate?

The features marked as `1` for *Midnight Chase* will dominate the profile. In this dataset, those features are **Action**.

### What could go wrong?

The recommender may become too narrow because one movie is not enough evidence to understand the user's full taste. It could repeatedly recommend Action-oriented movies even if the user actually enjoys several other genres.

### Is one rating enough?

No. One rating provides very limited information. More ratings across different types of movies would produce a more reliable user profile.

## 23. Add a Unique Movie

The practical requires each student to add a unique movie with their own title and feature combination.

For this project, the added movie is **Digital Shadows**, a SciFi/Action movie.

In [ ]:
new_movie = {
    "movie_id": 16,
    "title": "Digital Shadows",
    "Action": 1,
    "Comedy": 0,
    "Drama": 0,
    "Romance": 0,
    "SciFi": 1
}

movies = pd.concat(
    [movies, pd.DataFrame([new_movie])],
    ignore_index=True
)

movies.tail()

### Why this movie was added

*Digital Shadows* combines **Action** and **SciFi**, which makes it especially relevant to User A because those are the user's strongest preferences in the example profile.

The new movie is added before the final personal-user experiment so that the recommender can consider it as a candidate.

## 24. Create Your Own User

The lab requires a personal user profile with at least four rated movies.

The following ratings create a different preference pattern from User A and User B.

> These ratings are the example personal-user profile used in this submission. They can be changed to represent a different preference profile.

In [ ]:
my_user = pd.DataFrame({
    "movie_id": [2, 5, 10, 12],
    "rating": [5, 5, 3, 2]
})

my_user

## 25. Generate Top 5 Recommendations for My User

The recommender is now applied to the personal user profile, requesting five unseen recommendations.

In [ ]:
my_recommendations = recommend_movies(
    movies,
    my_user,
    features,
    n=5
)

my_recommendations

## 26. Interpret My User Profile

To explain the recommendations, first calculate the user's feature profile directly from the four ratings.

In [ ]:
my_user_movies = my_user.merge(
    movies,
    on="movie_id"
)

my_rated_features = my_user_movies[features]

my_weighted_features = my_rated_features.mul(
    my_user_movies["rating"],
    axis=0
)

my_user_profile = my_weighted_features.sum()
my_user_profile.sort_values(ascending=False)

### Interpretation Questions

**8. What are your user's two strongest preferences?**  
The strongest preferences are determined from the two highest values in the profile displayed above.

**9. Which three movies received the highest recommendation scores?**  
They are the first three rows of `my_recommendations`.

**10. Why did the top movie rank first?**  
The top movie has the strongest overlap with the features that have the highest weights in the user's profile.

**11. Did any recommendation surprise you?**  
A recommendation may be surprising when a movie receives a high score because of one or two shared features even though its overall genre combination is different from what the user normally watches.

**12. What important information is the model ignoring?**  
The model ignores information such as actors, directors, release year, language, descriptions, storyline, popularity, and other contextual information because only five structured genre features are used.

## 27. Limitations of the Recommender

This is a deliberately simple content-based recommender, so it has several limitations.

### 1. Limited Features

Only five genre features are used. Real recommendation systems normally use much richer item metadata.

### 2. Missing Metadata

The model ignores actors, directors, release year, language, descriptions, storyline, and other information that could help distinguish movies.

### 3. Overspecialization

Because recommendations are based on the user's existing preferences, the system can repeatedly recommend items with similar features. This may reduce discovery of new types of content.

### 4. Cold-Start User Problem

A new user with no ratings has no meaningful preference profile. Even one rating can be too little to understand a user's interests reliably.

### 5. Dependence on Metadata Quality

If the genre labels or other item features are incorrect or incomplete, the recommendation scores will also be affected.

### 6. Simple Rating Weighting

The method assumes that a higher rating directly means a stronger positive preference. It does not model more complex rating behaviour or differences between users.

### 7. No Collaborative Information

The model does not use the behaviour of other users. Two users can receive recommendations independently even when their tastes are similar.

## 28. Content-Based Filtering Workflow

The complete workflow implemented in this notebook is:

**Item Metadata → Item Profiles → User Ratings → User Profile → Candidate Scores → Remove Seen Items → Rank Items → Top-N Recommendations**

The system therefore follows the structure required for the Week 3 practical and uses structured features rather than TF-IDF or cosine similarity.

## 29. Final Question

### If two users have watched exactly the same movies but gave them different ratings, should this recommender produce the same recommendations?

**No.**

The recommender does not only look at which movies a user has watched. It also uses the ratings to weight the movie features.

If two users watched the same movies but rated them differently, their weighted feature totals can be different. This produces different normalized user profiles, which in turn produces different recommendation scores for unseen movies.

Therefore, the same viewing history does not necessarily produce the same recommendations when the ratings are different.

## 30. Conclusion

This practical demonstrated how a simple content-based recommender can be built using structured movie features and user ratings.

The main stages were:

1. Represent movies using binary genre features.
2. Build an item-feature matrix.
3. Combine movie features with a user's ratings.
4. Weight features according to rating strength.
5. Build and normalize a user preference profile.
6. Score every movie according to the user's preferences.
7. Remove movies the user has already watched.
8. Rank the remaining movies.
9. Generate Top-N recommendations.
10. Test the approach with multiple users and a unique movie.

The approach is easy to understand and implement, but it is limited by the small number of features and the absence of richer metadata and collaborative information. More advanced recommendation techniques can address some of these limitations in later practicals.